# Open-Meteo Weather Forecasting with Chronos-Bolt

This notebook loads hourly weather data from Open-Meteo, cleans it, and uses a pretrained Chronos-Bolt time-series model to forecast the next 24 hours.

Chronos is used in a zero-shot univariate setup, so each weather variable is forecast separately. Wind direction is handled as a circular variable by forecasting sine/cosine components instead of raw degrees.

In [25]:
from __future__ import annotations

import math
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from chronos import ChronosBoltPipeline

## Configuration

In [26]:
LATITUDE = 30.055128782798967
LONGITUDE = 31.357082654123047
START_DATE = "2026-05-01"
END_DATE = "2026-06-03"

WEATHER_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
]

PREDICTION_LENGTH = 24
MODEL_ID = "amazon/chronos-bolt-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

## Load Open-Meteo Data

In [27]:
def fetch_open_meteo_hourly(
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
    hourly_columns: Iterable[str],
) -> pd.DataFrame:
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_columns),
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if "hourly" not in data:
        reason = data.get("reason", "No reason returned by Open-Meteo")
        raise ValueError(f"Open-Meteo response does not contain hourly data: {reason}")

    df = pd.DataFrame(data["hourly"])
    if df.empty:
        raise ValueError("Open-Meteo returned an empty hourly dataframe.")

    df["time"] = pd.to_datetime(df["time"])
    df = df.set_index("time").sort_index()
    return df


raw_df = fetch_open_meteo_hourly(
    latitude=LATITUDE,
    longitude=LONGITUDE,
    start_date=START_DATE,
    end_date=END_DATE,
    hourly_columns=WEATHER_COLUMNS,
)

raw_df.head()

,temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,wind_direction_10m
time,,,,,
2026-05-01 00:00:00,16.8,80,997.5,10.0,75
2026-05-01 01:00:00,16.3,83,997.1,9.6,86
2026-05-01 02:00:00,15.8,83,996.9,7.4,76
2026-05-01 03:00:00,15.2,83,996.7,7.0,67
2026-05-01 04:00:00,15.4,79,997.2,7.9,57


In [1]:
import torch 

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [3]:
torch.__version__

'2.12.0+cpu'

## Clean and Validate Hourly Data

In [28]:
def clean_hourly_weather(df: pd.DataFrame) -> pd.DataFrame:
    missing_columns = sorted(set(WEATHER_COLUMNS) - set(df.columns))
    if missing_columns:
        raise ValueError(f"Missing expected weather columns: {missing_columns}")

    cleaned = df[WEATHER_COLUMNS].copy()
    cleaned = cleaned.apply(pd.to_numeric, errors="coerce")
    cleaned = cleaned[~cleaned.index.duplicated(keep="last")].sort_index()
    cleaned = cleaned.asfreq("h")

    missing_before = int(cleaned.isna().sum().sum())
    cleaned = cleaned.interpolate(method="time").ffill().bfill()
    missing_after = int(cleaned.isna().sum().sum())

    if missing_after:
        raise ValueError("Weather dataframe still contains missing values after interpolation.")

    print(f"Rows: {len(cleaned)}")
    print(f"Date range: {cleaned.index.min()} -> {cleaned.index.max()}")
    print(f"Missing values filled: {missing_before}")
    return cleaned


df = clean_hourly_weather(raw_df)
df.head()

Rows: 816
Date range: 2026-05-01 00:00:00 -> 2026-06-03 23:00:00
Missing values filled: 0


,temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,wind_direction_10m
time,,,,,
2026-05-01 00:00:00,16.8,80,997.5,10.0,75
2026-05-01 01:00:00,16.3,83,997.1,9.6,86
2026-05-01 02:00:00,15.8,83,996.9,7.4,76
2026-05-01 03:00:00,15.2,83,996.7,7.0,67
2026-05-01 04:00:00,15.4,79,997.2,7.9,57


## Prepare Wind Direction as a Circular Variable

In [29]:
def add_wind_direction_components(df: pd.DataFrame) -> pd.DataFrame:
    prepared = df.copy()
    radians = np.deg2rad(prepared["wind_direction_10m"] % 360)
    prepared["wind_direction_sin"] = np.sin(radians)
    prepared["wind_direction_cos"] = np.cos(radians)
    return prepared.drop(columns=["wind_direction_10m"])


def direction_from_components(sin_values: np.ndarray, cos_values: np.ndarray) -> np.ndarray:
    degrees = np.rad2deg(np.arctan2(sin_values, cos_values))
    return (degrees + 360) % 360


model_input_df = add_wind_direction_components(df)
model_input_df.head()

,temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,wind_direction_sin,wind_direction_cos
time,,,,,,
2026-05-01 00:00:00,16.8,80,997.5,10.0,0.965926,0.258819
2026-05-01 01:00:00,16.3,83,997.1,9.6,0.997564,0.069756
2026-05-01 02:00:00,15.8,83,996.9,7.4,0.970296,0.241922
2026-05-01 03:00:00,15.2,83,996.7,7.0,0.920505,0.390731
2026-05-01 04:00:00,15.4,79,997.2,7.9,0.838671,0.544639


## Load Pretrained Chronos-Bolt

In [30]:
pipeline = ChronosBoltPipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    dtype=TORCH_DTYPE,
)

pipeline

## Forecast the Next 24 Hours

In [ ]:
def forecast_series(
    pipeline: ChronosBoltPipeline,
    series: pd.Series,
    prediction_length: int,
) -> pd.DataFrame:
    context = torch.tensor(series.to_numpy(dtype=np.float32))
    quantiles, mean = pipeline.predict_quantiles(
        [context],
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
    )

    q = quantiles[0].detach().cpu().numpy()
    mean_values = mean[0].detach().cpu().numpy()
    return pd.DataFrame(
        {
            "lower_10": q[:, 0],
            "median": q[:, 1],
            "upper_90": q[:, 2],
            "mean": mean_values,
        }
    )


future_index = pd.date_range(
    start=model_input_df.index[-1] + pd.Timedelta(hours=1),
    periods=PREDICTION_LENGTH,
    freq="h",
)

forecast_parts = {}
for column in model_input_df.columns:
    forecast_parts[column] = forecast_series(pipeline, model_input_df[column], PREDICTION_LENGTH)

forecast_df = pd.concat(forecast_parts, axis=1)
forecast_df.index = future_index
forecast_df.head()

## Build Final Weather Forecast Table

In [ ]:
def build_weather_forecast_table(forecast_df: pd.DataFrame) -> pd.DataFrame:
    result = pd.DataFrame(index=forecast_df.index)

    for column in ["temperature_2m", "relative_humidity_2m", "surface_pressure", "wind_speed_10m"]:
        result[column] = forecast_df[(column, "median")]
        result[f"{column}_lower_10"] = forecast_df[(column, "lower_10")]
        result[f"{column}_upper_90"] = forecast_df[(column, "upper_90")]

    result["relative_humidity_2m"] = result["relative_humidity_2m"].clip(0, 100)
    result["relative_humidity_2m_lower_10"] = result["relative_humidity_2m_lower_10"].clip(0, 100)
    result["relative_humidity_2m_upper_90"] = result["relative_humidity_2m_upper_90"].clip(0, 100)
    result["wind_speed_10m"] = result["wind_speed_10m"].clip(lower=0)
    result["wind_speed_10m_lower_10"] = result["wind_speed_10m_lower_10"].clip(lower=0)
    result["wind_speed_10m_upper_90"] = result["wind_speed_10m_upper_90"].clip(lower=0)

    result["wind_direction_10m"] = direction_from_components(
        forecast_df[("wind_direction_sin", "median")].to_numpy(),
        forecast_df[("wind_direction_cos", "median")].to_numpy(),
    )
    return result


weather_forecast_24h = build_weather_forecast_table(forecast_df)
weather_forecast_24h.round(2)

## Plot Forecasts

In [ ]:
def plot_forecast(history: pd.DataFrame, forecast: pd.DataFrame, column: str, history_hours: int = 7 * 24) -> None:
    plt.figure(figsize=(12, 4))
    plt.plot(history.index[-history_hours:], history[column].iloc[-history_hours:], label="history")
    plt.plot(forecast.index, forecast[column], label="forecast", color="tab:orange")

    lower_col = f"{column}_lower_10"
    upper_col = f"{column}_upper_90"
    if lower_col in forecast.columns and upper_col in forecast.columns:
        plt.fill_between(
            forecast.index,
            forecast[lower_col],
            forecast[upper_col],
            color="tab:orange",
            alpha=0.2,
            label="10%-90% interval",
        )

    plt.title(f"Next {PREDICTION_LENGTH} hours: {column}")
    plt.xlabel("time")
    plt.ylabel(column)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


for column in WEATHER_COLUMNS:
    plot_forecast(df, weather_forecast_24h, column)

## Rolling Backtest: 7-Day Context to Forecast the Next Week

This evaluates the model over 7 daily 24-hour forecasts from `2026-05-28` to `2026-06-03`. For each forecast day, the model only sees the previous 7 days, forecasts the next 24 hours, then compares the forecast with the actual Open-Meteo values.

In [31]:
BACKTEST_START = pd.Timestamp("2026-05-28 00:00:00")
BACKTEST_END = pd.Timestamp("2026-06-03 23:00:00")
CONTEXT_LENGTH = 7 * 24


def angular_error_degrees(predicted: pd.Series, actual: pd.Series) -> pd.Series:
    return ((predicted - actual + 180) % 360) - 180


def calculate_backtest_metrics(predictions: pd.DataFrame, actual: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in WEATHER_COLUMNS:
        if column == "wind_direction_10m":
            error = angular_error_degrees(predictions[column], actual[column])
            mape = np.nan
        else:
            error = predictions[column] - actual[column]
            non_zero_actual = actual[column].replace(0, np.nan)
            mape = float((error.abs() / non_zero_actual).mean() * 100)

        rows.append(
            {
                "column": column,
                "mae": float(error.abs().mean()),
                "rmse": float(math.sqrt((error**2).mean())),
                "bias": float(error.mean()),
                "mape_percent": mape,
            }
        )
    return pd.DataFrame(rows).set_index("column")


def run_daily_rolling_backtest() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if BACKTEST_START not in model_input_df.index or BACKTEST_END not in model_input_df.index:
        raise ValueError(
            f"Backtest window {BACKTEST_START} to {BACKTEST_END} is not fully available. "
            f"Available range: {model_input_df.index.min()} to {model_input_df.index.max()}"
        )

    daily_predictions = []
    daily_metrics = []
    forecast_starts = pd.date_range(BACKTEST_START, BACKTEST_END, freq="24h")

    for forecast_start in forecast_starts:
        forecast_end = forecast_start + pd.Timedelta(hours=PREDICTION_LENGTH - 1)
        context_start = forecast_start - pd.Timedelta(hours=CONTEXT_LENGTH)
        context_end = forecast_start - pd.Timedelta(hours=1)
        train_input = model_input_df.loc[context_start:context_end]
        actual_block = df.loc[forecast_start:forecast_end]

        if len(train_input) != CONTEXT_LENGTH:
            raise ValueError(
                f"Expected {CONTEXT_LENGTH} context rows for {forecast_start}, got {len(train_input)}. "
                f"Context window: {context_start} to {context_end}"
            )

        if len(actual_block) != PREDICTION_LENGTH:
            raise ValueError(f"Expected 24 actual rows for {forecast_start}, got {len(actual_block)}.")

        backtest_parts = {}
        for column in train_input.columns:
            backtest_parts[column] = forecast_series(pipeline, train_input[column], PREDICTION_LENGTH)

        forecast_block_df = pd.concat(backtest_parts, axis=1)
        forecast_block_df.index = actual_block.index
        forecast_block = build_weather_forecast_table(forecast_block_df)
        forecast_block["forecast_start"] = forecast_start
        daily_predictions.append(forecast_block)

        metrics = calculate_backtest_metrics(forecast_block, actual_block)
        metrics["forecast_start"] = forecast_start
        daily_metrics.append(metrics.reset_index())

    predictions = pd.concat(daily_predictions).sort_index()
    actual = df.loc[BACKTEST_START:BACKTEST_END]
    metrics_by_column = calculate_backtest_metrics(predictions, actual)
    metrics_by_day = pd.concat(daily_metrics).set_index(["forecast_start", "column"])
    return predictions, metrics_by_column, metrics_by_day


backtest_predictions, backtest_metrics, backtest_daily_metrics = run_daily_rolling_backtest()
display(backtest_metrics.round(3))
display(backtest_daily_metrics.round(3))

,mae,rmse,bias,mape_percent
column,,,,
temperature_2m,0.943,1.211,-0.511,3.317
relative_humidity_2m,5.239,6.697,2.958,16.807
surface_pressure,0.663,0.797,0.126,0.066
wind_speed_10m,2.615,3.537,0.163,39.922
wind_direction_10m,28.138,40.755,3.635,NaN


mae    rmse    bias  mape_percent
forecast_start column                                                    
2026-05-28     temperature_2m         0.469   0.601  -0.136         1.765
               relative_humidity_2m   7.264   8.251   5.113        22.362
               surface_pressure       0.562   0.620   0.537         0.056
               wind_speed_10m         3.504   4.440  -1.272        60.606
               wind_direction_10m    44.820  49.698  20.944           NaN
2026-05-29     temperature_2m         0.642   0.748   0.125         2.348
               relative_humidity_2m   3.760   4.696   2.209         9.337
               surface_pressure       0.850   0.945  -0.847         0.085
               wind_speed_10m         2.419   3.063   1.206        38.716
               wind_direction_10m    26.732  38.460   7.959           NaN
2026-05-30     temperature_2m         1.066   1.219  -0.124         4.189
               relative_humidity_2m   5.703   6.877  -1.942        13.808
               surface_pressure       0.973   1.112   0.973         0.097
               wind_speed_10m         1.780   2.096   0.040        25.500
               wind_direction_10m    12.214  17.388   4.402           NaN
2026-05-31     temperature_2m         0.360   0.412   0.012         1.357
               relative_humidity_2m   5.135   6.919   4.186        16.351
               surface_pressure       0.652   0.810  -0.652         0.065
               wind_speed_10m         1.230   1.565  -0.479        13.869
               wind_direction_10m    16.240  21.420 -10.909           NaN
2026-06-01     temperature_2m         1.037   1.189  -0.921         3.686
               relative_humidity_2m   5.075   6.984   2.059        16.655
               surface_pressure       0.538   0.694   0.520         0.054
               wind_speed_10m         1.655   2.300  -0.502        21.256
               wind_direction_10m    19.950  22.948 -16.069           NaN
2026-06-02     temperature_2m         1.434   1.727  -1.428         4.815
               relative_humidity_2m   4.675   6.395   4.385        17.151
               surface_pressure       0.416   0.519   0.005         0.042
               wind_speed_10m         3.370   4.113   2.260        52.356
               wind_direction_10m    34.993  55.070  -4.468           NaN
2026-06-03     temperature_2m         1.596   1.815  -1.102         5.061
               relative_humidity_2m   5.060   6.248   4.699        21.988
               surface_pressure       0.650   0.723   0.342         0.065
               wind_speed_10m         4.350   5.426  -0.111        67.152
               wind_direction_10m    42.014  57.941  23.588           NaN